# Exploração do SIH/SUS

Esse notebook realiza a exploração inicial dos dados de internações hospitalares do SIH/SUS.

Nesta etapa serão analisados:

- arquivos de AIH Reduzida (`RD`)
- estrutura das colunas
- quantidade de registros
- dados ausentes
- tipos das variáveis
- variáveis relacionadas a hospitais, permanência, UTI, valores e diagnósticos.

A análise será feita por ano, carregando automaticamente todos os arquivos mensais disponíveis.

## 1. Importação das bibliotecas

In [1]:
from pathlib import Path
import pandas as pd
from pysus.api.extensions import ExtensionFactory

## 2. Configuração da análise

O notebook procura automaticamente todos os arquivos `RD` disponíveis na pasta correspondente.

In [2]:
ANO = 2021
pasta_sih = Path(f"../data/raw/sih/{ANO}")
arquivos = sorted(pasta_sih.glob("RDGO*.dbc"))

print(f"Ano analisado: {ANO}")
print(f"Arquivos encontrados: {len(arquivos)}")
for arquivo in arquivos:
    print(arquivo.name)

Ano analisado: 2021
Arquivos encontrados: 12
RDGO2101.dbc
RDGO2102.dbc
RDGO2103.dbc
RDGO2104.dbc
RDGO2105.dbc
RDGO2106.dbc
RDGO2107.dbc
RDGO2108.dbc
RDGO2109.dbc
RDGO2110.dbc
RDGO2111.dbc
RDGO2112.dbc


## 3. Validação dos arquivos mensais

Antes de carregar os dados, verificamos se os 12 meses do ano estão disponíveis

In [3]:
meses_encontrados = [arquivo.stem[-2:] for arquivo in arquivos]
print("Meses encontrados:", meses_encontrados)
if len(arquivos) == 12:
    print("Todos os meses estão disponíveis.")
else:
    print("Atenção: o ano não possui 12 arquivos.")

Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Todos os meses estão disponíveis.


## 4. Leitura dos arquivos DBC

Os arquivos mensais serão carregados e reunidos em um único DataFrame para facilitar a análise do ano.

In [4]:
async def carregar_dbc(caminho):
    extensao = await ExtensionFactory.instantiate(caminho)
    df = await extensao.load()

    # Guarda o nome do arquivo de origem
    df["ARQUIVO_ORIGEM"] = caminho.name

    return df

In [5]:
dataframes = []
for arquivo in arquivos:
    print(f"Carregando {arquivo.name}...")
    df_mes = await carregar_dbc(arquivo)
    dataframes.append(df_mes)

df_sih = pd.concat(dataframes, ignore_index=True)

print("\nCarga concluída.")

Carregando RDGO2101.dbc...
Carregando RDGO2102.dbc...
Carregando RDGO2103.dbc...
Carregando RDGO2104.dbc...
Carregando RDGO2105.dbc...
Carregando RDGO2106.dbc...
Carregando RDGO2107.dbc...
Carregando RDGO2108.dbc...
Carregando RDGO2109.dbc...
Carregando RDGO2110.dbc...
Carregando RDGO2111.dbc...
Carregando RDGO2112.dbc...

Carga concluída.


## 5. Dimensão do dataset

Nesta etapa verificamos quantas linhas e colunas existem após reunir todos os meses do ano

In [6]:
linhas, colunas = df_sih.shape

print(f"Registros de AIH: {linhas:,}")
print(f"Colunas: {colunas}")

Registros de AIH: 343,686
Colunas: 114


## 6. Visualização inicial

In [7]:
df_sih.head()

,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9,ARQUIVO_ORIGEM
0,520000,2021,01,03,,5220103703310,1,75920000,521930,19930807,...,0,0,0,0,0,0,0,0,0,RDGO2101.dbc
1,520000,2021,01,03,,5220103703320,1,75920000,521930,19880624,...,0,0,0,0,0,0,0,0,0,RDGO2101.dbc
2,520000,2021,01,03,,5220103703353,1,75920000,521930,19581017,...,0,0,0,0,0,0,0,0,0,RDGO2101.dbc
3,520000,2021,01,03,,5220103703397,1,75930000,521300,19991216,...,0,0,0,0,0,0,0,0,0,RDGO2101.dbc
4,520000,2021,01,03,,5220103703419,1,75920000,521930,19790701,...,0,0,0,0,0,0,0,0,0,RDGO2101.dbc


## 7. Colunas disponíveis

In [8]:
for coluna in df_sih.columns:
    print(coluna)

UF_ZI
ANO_CMPT
MES_CMPT
ESPEC
CGC_HOSP
N_AIH
IDENT
CEP
MUNIC_RES
NASC
SEXO
UTI_MES_IN
UTI_MES_AN
UTI_MES_AL
UTI_MES_TO
MARCA_UTI
UTI_INT_IN
UTI_INT_AN
UTI_INT_AL
UTI_INT_TO
DIAR_ACOM
QT_DIARIAS
PROC_SOLIC
PROC_REA
VAL_SH
VAL_SP
VAL_SADT
VAL_RN
VAL_ACOMP
VAL_ORTP
VAL_SANGUE
VAL_SADTSR
VAL_TRANSP
VAL_OBSANG
VAL_PED1AC
VAL_TOT
VAL_UTI
US_TOT
DT_INTER
DT_SAIDA
DIAG_PRINC
DIAG_SECUN
COBRANCA
NATUREZA
NAT_JUR
GESTAO
RUBRICA
IND_VDRL
MUNIC_MOV
COD_IDADE
IDADE
DIAS_PERM
MORTE
NACIONAL
NUM_PROC
CAR_INT
TOT_PT_SP
CPF_AUT
HOMONIMO
NUM_FILHOS
INSTRU
CID_NOTIF
CONTRACEP1
CONTRACEP2
GESTRISCO
INSC_PN
SEQ_AIH5
CBOR
CNAER
VINCPREV
GESTOR_COD
GESTOR_TP
GESTOR_CPF
GESTOR_DT
CNES
CNPJ_MANT
INFEHOSP
CID_ASSO
CID_MORTE
COMPLEX
FINANC
FAEC_TP
REGCT
RACA_COR
ETNIA
SEQUENCIA
REMESSA
AUD_JUST
SIS_JUST
VAL_SH_FED
VAL_SP_FED
VAL_SH_GES
VAL_SP_GES
VAL_UCI
MARCA_UCI
DIAGSEC1
DIAGSEC2
DIAGSEC3
DIAGSEC4
DIAGSEC5
DIAGSEC6
DIAGSEC7
DIAGSEC8
DIAGSEC9
TPDISEC1
TPDISEC2
TPDISEC3
TPDISEC4
TPDISEC5
TPDISEC6
TPDISEC7
TPDISE

## 8. Variáveis de interesse

O SIH/SUS possui mais de 100 campos. Nesse sentido para o trabalho, inicialmente serão observadas as variáveis relacionadas a período, hospital, município, permanência, UTI, valores, diagnóstico e perfil do paciente.

In [9]:
colunas_interesse = [
    "ANO_CMPT",
    "MES_CMPT",
    "N_AIH",
    "IDENT",
    "SEQ_AIH5",
    "CNES",
    "MUNIC_RES",
    "MUNIC_MOV",
    "DT_INTER",
    "DT_SAIDA",
    "DIAS_PERM",
    "UTI_MES_TO",
    "UTI_INT_TO",
    "VAL_TOT",
    "VAL_UTI",
    "DIAG_PRINC",
    "IDADE",
    "SEXO",
    "MORTE",
    "ESPEC",
    "CAR_INT",
    "COMPLEX",
]

colunas_existentes = [
    coluna
    for coluna in colunas_interesse
    if coluna in df_sih.columns
]

df_sih[colunas_existentes].head()

,ANO_CMPT,MES_CMPT,N_AIH,IDENT,SEQ_AIH5,CNES,MUNIC_RES,MUNIC_MOV,DT_INTER,DT_SAIDA,...,UTI_INT_TO,VAL_TOT,VAL_UTI,DIAG_PRINC,IDADE,SEXO,MORTE,ESPEC,CAR_INT,COMPLEX
0,2021,01,5220103703310,1,000,6665322,521930,521930,2020-10-29,20201101,...,0,485.78,0.00,K929,27,1,0,03,02,02
1,2021,01,5220103703320,1,000,6665322,521930,521930,2020-10-31,20201103,...,0,242.88,0.00,R31,32,1,0,03,02,02
2,2021,01,5220103703353,1,000,6665322,521930,521930,2020-10-08,20201009,...,0,40.38,0.00,I743,61,1,0,03,02,02
3,2021,01,5220103703397,1,000,6665322,521300,521930,2020-10-28,20201102,...,0,503.85,0.00,C819,20,1,0,03,02,02
4,2021,01,5220103703419,1,000,6665322,521930,521930,2020-11-11,20201112,...,0,40.38,0.00,K922,41,1,1,03,02,02


## 9. Verificação de campos ausentes

Os arquivos do DATASUS podem representar valores ausentes como campos vazios e não necessariamente como `NaN`. Por isso, serão considerados tanto valores nulos quanto textos vazios.

In [10]:
resumo = pd.DataFrame(index=df_sih.columns)

resumo["tipo"] = df_sih.dtypes.astype(str)
resumo["nulos"] = df_sih.isna().sum()

# Conta textos vazios
resumo["vazios"] = df_sih.apply( lambda coluna: coluna.astype(str).str.strip().eq("").sum())

resumo["faltantes"] = ( resumo["nulos"] + resumo["vazios"])
resumo["faltantes_%"] = ( resumo["faltantes"] / len(df_sih) * 100).round(2)

resumo["valores_unicos"] = df_sih.nunique(dropna=True)
resumo.sort_values( "faltantes_%", ascending=False).head(30)

,tipo,nulos,vazios,faltantes,faltantes_%,valores_unicos
NUM_PROC,string,0,343686,343686,100.00,1
CPF_AUT,string,0,343686,343686,100.00,1
DIAGSEC7,string,0,343686,343686,100.00,1
DIAGSEC6,string,0,343685,343685,100.00,2
DIAGSEC9,string,0,343686,343686,100.00,1
DIAGSEC8,string,0,343686,343686,100.00,1
GESTOR_DT,string,0,343686,343686,100.00,1
INFEHOSP,string,0,343686,343686,100.00,1
DIAGSEC5,string,0,343681,343681,100.00,6
DIAGSEC4,string,0,343674,343674,100.00,13


## 10. Conversão de tipos

Os dados carregados do DBC chegam principalmente como texto.

Nesse sentido para realizar cálculos, algumas variáveis precisam ser convertidas para números e datas.

Os códigos de município, hospital e diagnóstico continuarão como texto para preservar zeros e identificadores.

In [11]:
df_eda = df_sih.copy()

colunas_numericas = [
    "ANO_CMPT",
    "MES_CMPT",
    "IDADE",
    "DIAS_PERM",
    "UTI_MES_TO",
    "UTI_INT_TO",
    "VAL_TOT",
    "VAL_UTI",
    "MORTE",
]

for coluna in colunas_numericas:
    if coluna in df_eda.columns:
        df_eda[coluna] = pd.to_numeric(
            df_eda[coluna],
            errors="coerce"
        )

In [12]:
colunas_data = [
    "DT_INTER",
    "DT_SAIDA",
    "NASC",
]

for coluna in colunas_data:
    if coluna in df_eda.columns:
        df_eda[coluna] = pd.to_datetime(
            df_eda[coluna],
            format="%Y%m%d",
            errors="coerce"
        )

## 11. Distribuição das AIHs

O campo `IDENT` será analisado antes de contar internações pois existem diferentes tipos de AIH e casos de continuidade.

In [13]:
df_eda["IDENT"].value_counts(dropna=False)

IDENT
1    335300
5      8386
Name: count, dtype: Int64

In [14]:
df_eda["SEQ_AIH5"].value_counts(dropna=False).head(20)

SEQ_AIH5
000    343686
Name: count, dtype: Int64

### 11.1 AIHs repetidas

Nesta etapa verificamos os números de AIH que aparecem mais de uma vez.

O objetivo é entender se esses registros estão relacionados as AIHs de continuidade antes de realizar qualquer remoção ou contagem definitiva de internações.

In [15]:
aih_repetidas = df_eda[df_eda["N_AIH"].duplicated(keep=False)].sort_values(["N_AIH", "ANO_CMPT", "MES_CMPT"])
aih_repetidas[
    [
        "N_AIH",
        "IDENT",
        "CNES",
        "DT_INTER",
        "DT_SAIDA",
        "DIAS_PERM",
        "MES_CMPT",
    ]
].head(30)

,N_AIH,IDENT,CNES,DT_INTER,DT_SAIDA,DIAS_PERM,MES_CMPT
1679,5208100141321,5,2535939,2008-01-01,2021-01-31,31,1
28581,5208100141321,5,2535939,2008-01-01,2021-02-28,28,2
55211,5208100141321,5,2535939,2008-01-01,2021-03-31,31,3
85103,5208100141321,5,2535939,2008-01-01,2021-04-30,30,4
110926,5208100141321,5,2535939,2008-01-01,2021-05-31,31,5
142225,5208100141321,5,2535939,2008-01-01,2021-06-30,30,6
170848,5208100141321,5,2535939,2008-01-01,2021-07-31,31,7
200264,5208100141321,5,2535939,2008-01-01,2021-08-31,31,8
231546,5208100141321,5,2535939,2008-01-01,2021-09-30,30,9
261756,5208100141321,5,2535939,2008-01-01,2021-10-31,31,10


In [16]:
print(f"Registros com N_AIH repetido: {len(aih_repetidas):,}")
print(f"AIHs distintas repetidas: {aih_repetidas['N_AIH'].nunique():,}")
print("\nTipos de AIH entre os registros repetidos:")
print(aih_repetidas["IDENT"].value_counts())

Registros com N_AIH repetido: 11,090
AIHs distintas repetidas: 3,361

Tipos de AIH entre os registros repetidos:
IDENT
5    8150
1    2940
Name: count, dtype: Int64


## 12. Registros por mês de competência

Esta análise verifica a distribuição dos registros de AIH pelos meses de competência do SIH/SUS. O mês de competência não representa necessariamente o mês de início da internação.

In [17]:
registros_mes = ( df_eda.groupby("MES_CMPT").size().reset_index(name="registros_aih"))
registros_mes

,MES_CMPT,registros_aih
0,1,27074
1,2,26092
2,3,29333
3,4,28169
4,5,29498
5,6,29606
6,7,30129
7,8,30616
8,9,30233
9,10,30188


## 13. Resumo mensal por competência

Aqui será criado um primeiro resumo com volume de registros, hospitais, municípios, permanência e valor registrado nas AIHs.

In [18]:
resumo_mensal = (
    df_eda
    .groupby(["ANO_CMPT", "MES_CMPT"])
    .agg(
        registros_aih=("N_AIH", "size"),
        hospitais=("CNES", "nunique"),
        municipios_residencia=("MUNIC_RES", "nunique"),
        permanencia_media=("DIAS_PERM", "mean"),
        valor_total_aih=("VAL_TOT", "sum"),
    )
    .reset_index()
)

resumo_mensal

,ANO_CMPT,MES_CMPT,registros_aih,hospitais,municipios_residencia,permanencia_media,valor_total_aih
0,2021,1,27074,250,436,5.045653,46516938.86
1,2021,2,26092,250,402,5.013146,44405057.98
2,2021,3,29333,255,416,5.202332,58042547.89
3,2021,4,28169,253,433,5.427562,63380067.96
4,2021,5,29498,254,383,4.990949,60776098.7
5,2021,6,29606,258,390,5.245558,64916492.41
6,2021,7,30129,255,409,5.307478,72752320.44
7,2021,8,30616,254,393,5.334008,72468238.27
8,2021,9,30233,255,389,5.263288,67294965.98
9,2021,10,30188,252,376,5.322413,58384780.12


## 14. Permanência hospitalar

Aqui será analisado inicialmente a distribuição dos dias de permanência registrados nas AIHs.

In [19]:
df_eda["DIAS_PERM"].describe()

count    343686.0
mean     5.160434
std       7.05567
min           0.0
25%           1.0
50%           3.0
75%           6.0
max         343.0
Name: DIAS_PERM, dtype: Float64

In [20]:
df_eda[
    [
        "N_AIH",
        "IDENT",
        "CNES",
        "DT_INTER",
        "DT_SAIDA",
        "DIAS_PERM",
    ]
].sort_values(
    "DIAS_PERM",
    ascending=False
).head(20)

,N_AIH,IDENT,CNES,DT_INTER,DT_SAIDA,DIAS_PERM
340853,5221103956398,1,2383942,2021-01-12,2021-12-21,343
297727,5221102538256,1,2361744,2021-01-03,2021-11-06,307
340090,5221104191721,1,2442302,2021-02-02,2021-12-04,305
314743,5221500201566,1,2340690,2021-01-17,2021-11-18,305
268153,5221103267655,1,2437139,2021-01-16,2021-10-18,275
27580,5220103764810,1,6665322,2020-07-01,2021-01-08,191
256680,5221103157380,1,2382466,2021-03-10,2021-09-13,187
229428,5221102783677,1,2437627,2021-02-03,2021-08-08,186
204450,5221103465193,1,2534916,2021-02-07,2021-08-11,185
312723,5221102879377,1,2442450,2021-04-02,2021-10-04,185


## 15. Uso de UTI

Nesta etapa verificamos os campos de utilização de UTI disponíveis no arquivo.

In [21]:
df_eda[
    [
        "UTI_MES_TO",
        "UTI_INT_TO",
        "VAL_UTI",
    ]
].describe()

,UTI_MES_TO,UTI_INT_TO,VAL_UTI
count,343686.0,343686.0,343686.0
mean,0.86691,0.012305,891.246445
std,3.602092,0.476404,4153.995866
min,0.0,0.0,0.0
25%,0.0,0.0,0.0
50%,0.0,0.0,0.0
75%,0.0,0.0,0.0
max,191.0,42.0,190400.0


### 15.1 Registros com utilização de UTI

Para complementar a análise, verificamos quantos registros apresentam utilização de UTI.

In [22]:
df_eda["USOU_UTI_MES"] = df_eda["UTI_MES_TO"] > 0

percentual_uti = (
    df_eda["USOU_UTI_MES"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

percentual_uti

USOU_UTI_MES
False    89.43
True     10.57
Name: proportion, dtype: Float64

In [23]:
df_eda.loc[ df_eda["USOU_UTI_MES"], "UTI_MES_TO"].describe()

count     36340.0
mean     8.198817
std      7.912032
min           1.0
25%           3.0
50%           6.0
75%          11.0
max         191.0
Name: UTI_MES_TO, dtype: Float64

## 16. Diagnósticos mais frequentes

O diagnóstico principal será utilizado posteriormente para comparar permanência, utilização de recursos e valores entre diferentes grupos de pacientes.

In [26]:
df_eda["DIAG_PRINC"].value_counts().head(20)

DIAG_PRINC
B342    52024
O800    12556
O820     5754
I64      4608
Z470     4073
J189     4062
N390     3975
I200     3457
O809     3180
I500     2853
K359     2762
S525     2695
Z039     2591
I219     2561
O821     2346
S822     2160
Z302     2146
I210     2069
O829     1770
F200     1662
Name: count, dtype: Int64

## 17. Hospitais com maior quantidade de registros

O código CNES permite identificar os estabelecimentos responsáveis pelas AIHs e será posteriormente utilizado para integrar SIH/SUS e CNES.

In [27]:
df_eda["CNES"].value_counts().head(20)

CNES
7743068    15687
2338262    13858
2506815    10807
2506858     9969
2338114     9933
2338424     9323
2338351     9311
2339196     9049
2361787     7831
2338734     6525
2339234     5846
3771962     5666
9680977     5423
5419662     5209
0086126     4777
2339110     4752
2534967     4609
2789647     4459
2673932     4210
2442612     4073
Name: count, dtype: Int64

## 18. Municípios de residência e atendimento

Os campos `MUNIC_RES` e `MUNIC_MOV` serão importantes para estudar deslocamentos de pacientes entre municípios.

In [28]:
fluxos = (
    df_eda
    .groupby(["MUNIC_RES", "MUNIC_MOV"])
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

fluxos.head(20)

,MUNIC_RES,MUNIC_MOV,registros
2450,520870,520870,73118
1489,520110,520110,16324
1547,520140,520140,15890
1563,520140,520870,13166
3695,521880,521880,9214
4084,522140,522140,5444
4068,522140,520870,5160
2890,521150,521150,4947
2033,520510,520510,4635
2355,520800,520800,4025


### 18.1 Fluxos entre municípios diferentes

Nesta etapa são considerados apenas os registros em que o município de residência é diferente do município de atendimento.

In [29]:
fluxos_externos = (
    df_eda[
        df_eda["MUNIC_RES"] != df_eda["MUNIC_MOV"]
    ]
    .groupby(["MUNIC_RES", "MUNIC_MOV"])
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

fluxos_externos.head(20)

,MUNIC_RES,MUNIC_MOV,registros
1553,520140,520870,13166
3907,522140,520870,5160
3818,522045,520870,3955
2363,520870,520140,2875
2436,520880,520870,2452
2589,521000,520870,1653
2295,520800,520870,1402
1489,520110,520870,1387
1881,520450,520870,1265
2664,521040,520870,1250


## 19. Duplicidades

Duplicidades serão apenas identificadas nesta etapa

Nenhum registro será removido antes de entendermos as regras de AIH e continuidade.

In [30]:
print("Linhas totalmente duplicadas:", df_eda.duplicated().sum())
print("Ocorrências adicionais de N_AIH:", df_eda["N_AIH"].duplicated().sum())

Linhas totalmente duplicadas: 0
Ocorrências adicionais de N_AIH: 7729


## 20. Conclusão

A exploração dos dados do SIH/SUS de 2021 permitiu identificar as principais características do conjunto de AIHs utilizado no projeto.

Os principais resultados observados foram:

- foram carregados os 12 arquivos mensais de 2021;
- o conjunto possui 343.686 registros de AIH;
- existem registros de AIH inicial (`IDENT = 1`) e de continuidade (`IDENT = 5`);
- algumas AIHs aparecem em mais de uma competência, indicando a necessidade de tratamento cuidadoso antes da contagem de internações;
- a permanência hospitalar apresenta grande variação e ocorrência de valores extremos;
- aproximadamente 10,57% dos registros apresentam valor positivo em `UTI_MES_TO`;
- existem fluxos relevantes de pacientes entre diferentes municípios;
- não foram encontradas linhas totalmente duplicadas.

Os resultados serão utilizados posteriormente na integração com os dados do CNES, permitindo relacionar internações hospitalares e capacidade de leitos.